# RAD · RSP_CONTROLLED_01_sd21_baseline_50steps

**Domanda sperimentale:** effetto di `RSP_CONTROLLED_01_sd21_baseline_50steps` su RAD-DINO, con confronto validation-only e tre seed indipendenti. Questo notebook non può leggere il test locked.


## 1–5 · Identità e regime

- Experiment ID logico: `raddino__RSP_CONTROLLED_01_sd21_baseline_50steps`
- Architettura: `raddino` (RAD-DINO)
- Dataset variant: `RSP_CONTROLLED_01_sd21_baseline_50steps`
- Regime: `controlled`
- Generatore: `01_sd21_baseline_50steps`


In [ ]:
from pathlib import Path
import json, os, sys

def find_project_root(start=Path.cwd()):
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "configs/classifier_experiment_matrix.json").is_file():
            return candidate
    raise FileNotFoundError("MammoDiffusion project root not found")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/utility"))
import classifier_experiment_runner as runner
import classifier_dataset_builder as datasets
import classifier_reporting as reporting
import classifier_interpretability as interpretability

ARCHITECTURE = 'raddino'
DATASET_VARIANT_ID = 'RSP_CONTROLLED_01_sd21_baseline_50steps'
EXPERIMENT_ID = 'raddino__RSP_CONTROLLED_01_sd21_baseline_50steps'
MODE = "auto"  # plan | auto | train | validate | metrics-only
RUN_SEEDS = [17, 42, 73]
ALLOW_RETRAIN = False
ALLOW_OVERWRITE_VERIFIED = False
RESUME = True
GENERATE_GRADCAM = True
GRADCAM_NUM_REAL_SAMPLES = 8
GRADCAM_NUM_SYNTHETIC_SAMPLES = 8
TINY_SMOKE = os.environ.get("MAMMO_CLASSIFIER_TINY") == "1"
DATASET_STATUS = 'READY'
DATASET_BLOCKER = None
assert MODE != "locked-test"
RESULTS_DIR = PROJECT_ROOT / "results/classifiers_matrix" / ARCHITECTURE / DATASET_VARIANT_ID / f"{ARCHITECTURE}_standard"
print(EXPERIMENT_ID, MODE, RUN_SEEDS, "resume=", RESUME, "tiny=", TINY_SMOKE)
print("results:", RESULTS_DIR)


## 4 — Provenance e configurazione

La cella seguente risolve i file canonici, firma il manifest e carica esclusivamente la validation reale. Le varianti bloccate restano documentate e non avviano training.


In [ ]:
registry = runner.load_dataset_variant_registry(PROJECT_ROOT)
variant = next(v for v in registry["variants"] if v["dataset_variant_id"] == DATASET_VARIANT_ID)
if DATASET_STATUS == "BLOCKED":
    dataset_summary = {"status": DATASET_STATUS, "blocker": DATASET_BLOCKER}
else:
    train_rows, validation_rows, dataset_manifest = datasets.build_training_and_validation_rows(PROJECT_ROOT, variant)
    dataset_summary = {
        "status": DATASET_STATUS, "counts": dataset_manifest["counts"],
        "train_samples": len(train_rows), "validation_samples": len(validation_rows),
        "dataset_signature": dataset_manifest["signature"],
        "validation_signature": dataset_manifest["validation_signature"],
        "validation_sources": sorted({row["source"] for row in validation_rows}),
    }
    if MODE != "plan":
        reporting.persist_dataset_summary(RESULTS_DIR, train_rows, validation_rows, dataset_manifest)
print(json.dumps(dataset_summary, indent=1))


## 5 — Composizione del dataset

Tabelle source × classe, conteggi e percentuali sono salvati in `results/.../dataset/`. Validation è esclusivamente reale.


## 6 — Esempi visivi del training set

La griglia usa un campionamento deterministico, senza selezione manuale degli esempi.


In [ ]:
if DATASET_STATUS != "BLOCKED":
    plan_figures = reporting.persist_plan_figures(RESULTS_DIR, has_synthetic=True, has_augmented=False)
    print(*plan_figures, sep="\n")


## 7 — Piano e stato resume

Il protocollo è unico per architettura. `auto` riusa un checkpoint verificato, altrimenti addestra, poi esegue validation e metriche. Ogni seed usa directory e checkpoint distinti.


In [ ]:
policy = runner.load_training_protocols(PROJECT_ROOT)["policies"][ARCHITECTURE]
plans = [runner.plan(PROJECT_ROOT, ARCHITECTURE, DATASET_VARIANT_ID, seed) for seed in RUN_SEEDS]
print(json.dumps({"policy": policy, "plans": plans}, indent=1))


## 8 — Costruzione modello

Il runner salva `model_summary.txt` e `model_architecture.json`; parametri trainable/frozen e input shape sono riportati senza dump verbosi.


## 9 — Training dei seed

Questa è l’unica cella operativa. Notebook e scheduler chiamano la stessa funzione condivisa. L’ensemble è la media delle probabilità dei seed 17/42/73 e la soglia è scelta sull’ensemble validation.


In [ ]:
if DATASET_STATUS == "BLOCKED":
    run_results = [{"status": "BLOCKED", "reason": DATASET_BLOCKER}]
else:
    run_results = runner.execute_configuration(
        PROJECT_ROOT, ARCHITECTURE, DATASET_VARIANT_ID,
        mode=MODE, run_seeds=RUN_SEEDS, tiny=TINY_SMOKE,
    )
print(json.dumps(run_results, indent=1, default=str))


## 10 — Curve di training

Loss, ROC-AUC, PR-AUC, learning rate, precision e recall sono salvati per seed e aggregati sotto `figures/`; le storie restano CSV.


In [ ]:
report_artifacts = reporting.render_complete_report(RESULTS_DIR)
print(json.dumps(report_artifacts, indent=1))


## 11 — Validation per seed

Le predizioni includono `patient_id`, `image_id`, label e probabilità. Nessuna cella importa il test.


## 12 — Ensemble validation

Media delle probabilità dei seed 17/42/73 dopo verifica rigorosa dell'allineamento; soglia congelata dalla validation.


## 13 — Metriche e risultati

ROC-AUC, PR-AUC, F1, sensitivity, specificity, PPV, NPV, balanced accuracy, MCC, accuracy, Brier, ECE e confusion matrix.


## 14 — Grafici validation

ROC, PR, calibration, confusion matrix e distribuzione delle probabilità sono visualizzati e salvati.


## 15 — Analisi degli errori

FP, FN, TP e TN sono selezionati deterministicamente dalla validation e salvati in CSV e figura.


## 16 — Grad-CAM / Gradient-based attribution

Tecnica: **attribuzione gradient-weighted dei patch token (non semplice attention map)**. Campioni reali sempre presenti; sintetici: **True**; augmented: **False**. Per campione vengono salvate mappe normalizzate per seed e media ensemble. Il manifest reale condiviso è `configs/interpretability_validation_samples.json`.


In [ ]:
policy_name = f"{ARCHITECTURE}_standard"
ensemble_path = (PROJECT_ROOT / "results/classifiers_matrix" / ARCHITECTURE /
                 DATASET_VARIANT_ID / policy_name / "ensemble/manifests/ensemble_validation_manifest.json")
print("ensemble:", ensemble_path, "exists=", ensemble_path.is_file())
if GENERATE_GRADCAM and MODE != "plan" and ensemble_path.is_file() and not TINY_SMOKE:
    attribution_status = interpretability.generate_configuration_attributions(
        PROJECT_ROOT, ARCHITECTURE, DATASET_VARIANT_ID, policy,
        seeds=RUN_SEEDS, limit=GRADCAM_NUM_REAL_SAMPLES)
    print(json.dumps(attribution_status, indent=1))
for seed in RUN_SEEDS:
    run_dir = runner.resolve_job(PROJECT_ROOT, ARCHITECTURE, DATASET_VARIANT_ID, seed)["run_dir"]
    print(seed, run_dir, sorted(p.name for p in run_dir.glob("*.json")) if run_dir.exists() else [])


## 17 — Riepilogo finale

Output canonici: `experiments/classifiers_matrix/<arch>/<variant>/<policy>/seed_<seed>/` e `results/classifiers_matrix/<arch>/<variant>/<policy>/`. Durata e picco VRAM sono diagnostiche operative; energia e CO₂ dei nuovi classificatori non sono tracciate. Il test locked non è importato né accessibile da questo notebook.
